# Validación y selección de modelos

En [El sobreajuste y la división en entrenamiento y prueba](overfitting_and_the_split.ipynb), la profundidad de un árbol se eligió mirando su error de prueba, con una advertencia: una vez se ha usado el conjunto de prueba para tomar una decisión, su error deja de ser una estimación honesta. Elegir *entre modelos* es el mismo tipo de decisión, y la más habitual: un modelo lineal o un bosque, diez vecinos o cincuenta. Cada elección hecha mirando el conjunto de prueba hace que su error parezca mejor que el error que tendrá el modelo elegido con datos nuevos.

## Elegir con el conjunto de prueba

¿Cuánto mejor, y con qué frecuencia? Un experimento con las viviendas de California: una pequeña muestra de 600 distritos, 400 para entrenar y 200 para probar, y 45 modelos candidatos (árboles de profundidad 1 a 15, y *k* vecinos más cercanos con *k* de 1 a 30). Se elige el candidato con el menor error de prueba, y luego se vuelve a medir con los 20 000 distritos que no participaron, que representan los datos nuevos que encontrará el modelo. Todo el experimento se repite con 30 muestras distintas.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = housing["MedHouseVal"]

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

candidates = (
    [DecisionTreeRegressor(max_depth=depth, random_state=0) for depth in range(1, 16)]
    + [make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k)) for k in range(1, 31)]
)
len(candidates)

45

`make_pipeline` une el escalador y los vecinos en un solo modelo, para que cada candidato se ajuste y se use con una sola llamada; [Viviendas de California con pipelines](../basic-workflows/california_with_pipelines.ipynb) lo explica. El experimento:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


def choose_on_the_test_set(seed):
    X_sample, X_fresh, y_sample, y_fresh = train_test_split(X, y, train_size=600, random_state=seed)
    X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=200, random_state=seed)
    test_errors, fresh_errors = [], []
    for candidate in candidates:
        candidate.fit(X_train, y_train)
        test_errors.append(mean_absolute_error(y_test, candidate.predict(X_test)))
        fresh_errors.append(mean_absolute_error(y_fresh, candidate.predict(X_fresh)))
    chosen = np.argmin(test_errors)
    return {"test MAE of the chosen model": test_errors[chosen], "its MAE on fresh data": fresh_errors[chosen]}


runs = 100_000 * pd.DataFrame([choose_on_the_test_set(seed) for seed in range(30)])
runs["optimism"] = runs["its MAE on fresh data"] - runs["test MAE of the chosen model"]
runs.describe().loc[["mean", "min", "50%", "max"]].round(0)

,test MAE of the chosen model,its MAE on fresh data,optimism
mean,51982.0,54166.0,2183.0
min,46375.0,51302.0,-2360.0
50%,52454.0,53582.0,1581.0
max,56448.0,60535.0,7222.0


In [ ]:
(runs["optimism"] > 0).mean()

np.float64(0.7333333333333333)

De media, el error de prueba del modelo elegido fue unos 2200 \$ menor que su error con datos nuevos, y fue menor en unas tres ejecuciones de cada cuatro. Ningún candidato aprendió de las filas de prueba: el sesgo viene solo de la elección. Entre 45 candidatos, el que parece mejor con 200 distritos concretos es en parte el que resultó encajar con esos distritos. Cuantos más candidatos se comparan, y cuantas menos filas de prueba hay, mayor es el efecto. En las ejecuciones donde la diferencia es negativa, el azar fue en sentido contrario: la estimación es optimista de media, no en cada ejecución.

## Tres conjuntos

La salida es mantener las decisiones lejos del conjunto de prueba. Los datos se dividen en tres:

| Conjunto | Se usa para | Con qué frecuencia |
| --- | --- | --- |
| **Entrenamiento** | ajustar los parámetros de cada candidato | cada vez que se entrena un candidato |
| **Validación** | comparar candidatos y elegir ajustes | tantas veces como haga falta |
| **Prueba** | medir el modelo elegido | una vez, al final |

El conjunto de validación asume el papel que hacía el conjunto de prueba en el experimento de arriba, y acaba igual de optimista, lo cual está bien: su trabajo es ordenar candidatos, no comunicar un número. El conjunto de prueba se deja sin tocar hasta que se han tomado todas las decisiones, así que su error sigue siendo una estimación honesta.

`train_test_split` divide en dos, así que se llama dos veces: primero se aparta el conjunto de prueba, y después el resto se divide en entrenamiento y validación. Una cuarta parte del 80 % restante es el 20 % del total:

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_validation, y_train, y_validation = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)
len(X_train), len(X_validation), len(X_test)

(12384, 4128, 4128)

## Comparar modelos con el conjunto de validación

Cuatro candidatos de distintos tipos, cada uno entrenado con el conjunto de entrenamiento y medido con el de validación. El escalador va en el pipeline de los vecinos, que dependen de distancias, y en el del modelo lineal, donde no cambia ninguna predicción pero hace comparables los pesos; se ajusta solo con las filas de entrenamiento, como en [Viviendas de California, paso a paso](../basic-workflows/california_by_hand.ipynb):

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

models = {
    "linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "k-nearest neighbours": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
    "decision tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "random forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
for model in models.values():
    model.fit(X_train, y_train)

In [ ]:
validation_errors = pd.DataFrame({
    name: {
        "training MAE ($)": 100_000 * mean_absolute_error(y_train, model.predict(X_train)),
        "validation MAE ($)": 100_000 * mean_absolute_error(y_validation, model.predict(X_validation)),
    }
    for name, model in models.items()
}).T.round(0)
validation_errors

,training MAE ($),validation MAE ($)
linear regression,52665.0,53333.0
k-nearest neighbours,39078.0,44462.0
decision tree,30039.0,42424.0
random forest,12415.0,33555.0


El [bosque aleatorio](../algorithms/tree-based/random_forests.ipynb) tiene el menor error de validación, unos 33 600 \$, frente a unos 53 300 \$ del modelo lineal (el modelo de [Viviendas de California, paso a paso](../basic-workflows/california_by_hand.ipynb), con más o menos el mismo error). También tiene, con mucho, la mayor diferencia entre sus errores de entrenamiento y de validación: unos 12 400 \$ frente a 33 600 \$. Una diferencia es una señal de sobreajuste, pero no descalifica a un modelo: lo que decide es el error con datos de los que no aprendió, y ahí el bosque sigue siendo el mejor de los cuatro. Su profundidad y su número de árboles también podrían ajustarse con este conjunto de validación; [Validación cruzada y búsqueda de hiperparámetros](cross_validation_and_search.ipynb) lo hace de forma sistemática.

In [ ]:
chosen_name = validation_errors["validation MAE ($)"].idxmin()
chosen_name

'random forest'

## Probar una sola vez

La elección está hecha, así que se puede usar el conjunto de prueba, una vez. El tipo de modelo elegido se vuelve a entrenar con los conjuntos de entrenamiento y validación juntos: las filas de validación ya han hecho su trabajo, y más datos suelen dar un modelo mejor. Su error de prueba es la estimación que hay que comunicar:

In [ ]:
from sklearn.base import clone

final_model = clone(models[chosen_name]).fit(X_rest, y_rest)
100_000 * mean_absolute_error(y_test, final_model.predict(X_test))

32773.10800872095

Unos 32 800 \$, cerca del error de validación del bosque y algo menor, en parte porque ahora ha aprendido de 16 512 distritos en lugar de 12 384, en parte porque los distritos de prueba son otros distritos; y con solo cuatro candidatos comparados, el error de validación tenía poco margen para ser optimista. Esta es la estimación que hay que comunicar para este modelo.

Una vez visto este número, no se cambia nada. Si decepciona y se cambia el modelo como respuesta, el conjunto de prueba se ha usado para tomar una decisión, y el siguiente error de prueba sería optimista de la misma forma que en el experimento de arriba. Una estimación honesta necesitaría entonces datos que aún no se hayan usado.

## El precio de un conjunto de validación

Un conjunto de validación son datos de los que el modelo no aprende, y con pocos datos eso duele dos veces: menos filas con las que entrenar, y un error de validación medido con pocas filas, que cambia notablemente según qué filas caigan en él. Con 600 distritos, como en el experimento, un conjunto de validación del 20 % serían 120 distritos. [Validación cruzada y búsqueda de hiperparámetros](cross_validation_and_search.ipynb) saca más partido de las mismas filas por turnos: cada parte de los datos hace una vez de conjunto de validación.